In [6]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import re
import time
import pandas as pd

visited_urls = set()
article_data = []

# List of URLs to exclude from crawling
excluded_urls = set([
    'http://altavz.com/author/', 
    'https://altavz.com/wp-login.php?redirect_to=https%3A%2F%2Faltavz.com%2F', 
    'https://altavz.com/registrarse/'
      # Add URLs here as strings, e.g., 'https://example.com/broken-link'
])

def is_relevant_url(url):
    """
    Check if the URL matches the pattern YYYY/MM/DD/title and is not in the excluded URLs.
    """
    pattern = r'\d{4}/\d{2}/\d{2}/[a-zA-Z0-9-]+'
    return re.search(pattern, url) and url not in excluded_urls

def extract_article_data(url):
    """
    Extract and return the title, main text, date, and source from a relevant article page.
    """
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # Extract the title
        title_tag = soup.find('h1', class_='entry-title')
        title = title_tag.get_text(strip=True) if title_tag else 'No title found'

        # Extract the main text
        main_content = soup.find_all('p')
        main_text = ' '.join(p.get_text(strip=True) for p in main_content)

        # Extract the date
        date_tag = soup.find('time')
        date = date_tag['datetime'] if date_tag and 'datetime' in date_tag.attrs else 'No date found'

        # Extract the source
        source_tag = soup.find('em')
        source = ''
        if source_tag:
            strong_tag = source_tag.find('strong')
            source = strong_tag.get_text(strip=True) if strong_tag else 'No source found'

        return {
            'url': url,
            'title': title,
            'main_text': main_text,
            'date': date,
            'source': source
        }
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data from {url}: {e}")
        return None

def crawl(url, depth=2):
    """
    Crawl a webpage, finding and following all relevant links within it, up to a specified depth.
    """
    if depth == 0 or url in visited_urls or url in excluded_urls:
        return

    try:
        # Fetch content from URL
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        visited_urls.add(url)
        print(f"Crawling: {url}")

        # Parse HTML content
        soup = BeautifulSoup(response.text, 'html.parser')

        # Find and process all relevant links
        for link in soup.find_all("a", href=True):
            # Join with base URL to get absolute link
            href = urljoin(url, link['href'])
            if is_relevant_url(href):
                # Extract and store article data from relevant pages
                article = extract_article_data(href)
                if article:
                    article_data.append(article)
                    print(f"Extracted data from {href}")

            # Recursive crawl for links within the same domain
            if urlparse(href).netloc == urlparse(url).netloc:
                crawl(href, depth - 1)

        # Wait to avoid overloading server
        time.sleep(1)

    except requests.exceptions.RequestException as e:
        print(f"Error crawling {url}: {e}")



In [7]:
# Starting point (replace with your initial URL)
start_url = "https://altavz.com/"
crawl(start_url, depth=2)

# Convert the article data to a DataFrame
df = pd.DataFrame(article_data)

# Deduplicate the DataFrame based on the 'url' column
df = df.drop_duplicates(subset='url', keep='first').reset_index(drop=True)

# Output the deduplicated DataFrame
print("Collected and deduplicated article data:")
print(df)

Crawling: https://altavz.com/
Crawling: https://altavz.com/#content
Extracted data from https://altavz.com/2024/10/01/en-vivo-toma-de-protesta-claudia-sheinbaum/
Extracted data from https://altavz.com/2024/09/30/a-horas-de-convertirse-en-la-primera-mujer-presidenta-de-mexico-claudia-sheinbaum-anuncio-este-lunes-una-reestructura-completa-del-gobierno-mexicano-en-el-primer-mes-de-2025-a-fin-de-disminuir-los-g/
Extracted data from https://altavz.com/2024/09/30/a-horas-de-convertirse-en-la-primera-mujer-presidenta-de-mexico-claudia-sheinbaum-anuncio-este-lunes-una-reestructura-completa-del-gobierno-mexicano-en-el-primer-mes-de-2025-a-fin-de-disminuir-los-g/
Extracted data from https://altavz.com/2024/09/29/reconoce-trump-que-se-enojo-con-harris-por-acusacion-de-que-no-hizo-nada-en-migracion/
Extracted data from https://altavz.com/2024/09/29/reconoce-trump-que-se-enojo-con-harris-por-acusacion-de-que-no-hizo-nada-en-migracion/
Extracted data from https://altavz.com/2024/09/28/tribunal-de-ju

In [9]:
df

,url,title,main_text,date,source
0,https://altavz.com/2024/10/01/en-vivo-toma-de-...,No title found,Sigue en vivo la toma de protesta de la presid...,2024-10-01T09:19:56-06:00,
1,https://altavz.com/2024/09/30/a-horas-de-conve...,No title found,A horas de convertirse en la primera mujer pre...,2024-09-30T06:56:03-06:00,(Fuente: EFE)
2,https://altavz.com/2024/09/29/reconoce-trump-q...,No title found,El expresidenteDonald Trumpse defendió este do...,2024-09-29T07:00:41-06:00,(Fuente: EFE)
3,https://altavz.com/2024/09/28/tribunal-de-just...,No title found,ElTribunal Estatal de Justicia Administrativad...,2024-09-28T07:02:32-06:00,(Fuente: López-Dóriga Digital)
4,https://altavz.com/2024/09/27/amlo-atribuye-a-...,No title found,El presidente mexicano Andrés Manuel López Obr...,2024-09-27T07:04:29-06:00,
...,...,...,...,...,...
144,https://altavz.com/2024/08/31/justicia-de-ee-u...,No title found,La Justicia estadounidense confirmó este viern...,2024-08-31T07:32:01-06:00,(Fuente: EFE)
145,https://altavz.com/2024/08/30/harris-tiene-est...,No title found,"La vicepresidenta y candidata demócrata, Kamal...",2024-08-30T18:02:35-06:00,(Fuente: EFE)
146,https://altavz.com/2024/08/30/harris-tiene-est...,No title found,"La vicepresidenta y candidata demócrata, Kamal...",2024-08-30T18:02:35-06:00,(Fuente: EFE)
147,https://altavz.com/2024/08/30/se-necesita-la-c...,No title found,Youtubers recibieron con aplausos y porras al ...,2024-08-30T18:00:49-06:00,(Fuente: López-Dóriga Digital)
